# 题目后的占位 token:anstok_v1

**问题**:答案句子的向量是所有 token 的平均,区分对错的关系词只占其中几个 token(flip_similarity_check:翻转关系后整句相似度 0.98)。预测端也是题目 token 的平均。

**做法**(底座 hardneg_v1,数据、步数、初始权重相同:6870 步,3 张卡,1 个 epoch):

- 题目后面加一个占位 token。
- **第一部分**:题目 token 的平均 → 预测长句答案(和 hardneg_v1 一样)。
- **第二部分**:占位 token 的输出 → 预测短答案("front-left"、"2.01 meters"、"right, front")。
- 两部分各有 InfoNCE 和同题错误答案的交叉熵,权重都是 1;长句和短答案走同一个 Y-encoder。

**评测**:长句答案按第一部分,短答案按占位 token。VSI-Bench 的选项是短词,按占位 token;题目平均的读法作参照。每种都有真场景、换场景、全零场景。

VSI-Bench 的换场景用修正后的题目顺序(`experiments/scneg_v1/data`,每道题都换到别的房间);hardneg_v1 也用它重评了。

## 总结

- **VSI-Bench test 4 任务平均 36.0,hardneg_v1 34.1**;debiased 34.2 对 34.3,没有差别。
- **场景差距(真 − 换)**:test 4.4 对 4.0,debiased 5.3 对 1.6。差距来自出现顺序(test 44.5 → 29.3)和相对距离(36.3 → 29.8)。
- **相对距离是主要收益**:test 36.3 对 26.2,debiased 35.9 对 25.8,并且换场景后掉回 22–30。
- **相对方向没有改善**:三种场景输入下相同(38.6 / 38.3 / 41.5);medium 一类在三种输入下都是 49.0,即输出固定答案。
- **路线规划下降**(24.4 对 33.3,只有 90 题)。
- **留出集**:长句部分与 hardneg_v1 相同(47.3 对 46.7),场景差距略大(3.9 对 2.6);短答案部分 37.0 对 35.1。空间想象和相对大小的短答案在随机水平。
- **占位 token 读长句不如题目平均**(42.0 对 47.3),题目平均读短答案也不如占位 token(34.7 对 37.0):两部分各自学了自己的目标。
- MMSI-Bench 未评测。

In [1]:
import json
import pandas as pd
load = lambda path: json.load(open(path))
A = load("../results/eval_final_otherscene.json")
H = load("../../hardneg_v1/results/eval_final_otherscene.json")
MODES = {"real": "真场景", "shuffled": "换场景", "zero": "全零场景"}
MC = {"相对距离": ["object_rel_distance"],
      "相对方向": ["object_rel_direction_easy", "object_rel_direction_medium", "object_rel_direction_hard"],
      "路线规划": ["route_planning"], "出现顺序": ["obj_appearance_order"]}
READ = {"anstok_v1 占位 token": (A, ""), "anstok_v1 题目平均": (A, "_pooled"), "hardneg_v1": (H, "")}

def tasks(res, split, suffix=""):
    mc = res[f"vsibench_{split}_posed_f8{suffix}"]
    out = {t: sum(100 * mc["type/" + k] for k in ks) / len(ks) for t, ks in MC.items()}
    return {**out, "4 任务平均": sum(out.values()) / 4}

## 1 VSI-Bench 选择题

带位姿的 ScanNet / ScanNet++ 部分:test 1,845 题,debiased 811 题。每个任务 90–620 题,3 个点以内的差别不算数。

In [2]:
pd.DataFrame({f"{split} {label}": {name: tasks(r[mode], split, s)["4 任务平均"] for name, (r, s) in READ.items()}
              for split in ("test", "debiased") for mode, label in MODES.items()}).T.round(1)

,anstok_v1 占位 token,anstok_v1 题目平均,hardneg_v1
test 真场景,36.0,30.0,34.1
test 换场景,31.6,24.2,30.1
test 全零场景,29.0,25.6,29.2
debiased 真场景,34.2,31.7,34.3
debiased 换场景,28.9,28.3,32.7
debiased 全零场景,27.7,32.0,31.1


按任务,test:

In [3]:
pd.DataFrame({f"{name} {MODES[m]}": tasks(r[m], "test", s) for name, (r, s) in READ.items() if "题目平均" not in name
              for m in MODES}).round(1)

,anstok_v1 占位 token 真场景,anstok_v1 占位 token 换场景,anstok_v1 占位 token 全零场景,hardneg_v1 真场景,hardneg_v1 换场景,hardneg_v1 全零场景
相对距离,36.3,29.8,24.9,26.2,23.2,22.5
相对方向,38.6,38.3,41.5,35.9,34.7,32.1
路线规划,24.4,28.9,25.6,33.3,30.0,32.2
出现顺序,44.5,29.3,24.3,40.9,32.5,29.9
4 任务平均,36.0,31.6,29.0,34.1,30.1,29.2


按任务,debiased:

In [4]:
pd.DataFrame({f"{name} {MODES[m]}": tasks(r[m], "debiased", s) for name, (r, s) in READ.items() if "题目平均" not in name
              for m in MODES}).round(1)

,anstok_v1 占位 token 真场景,anstok_v1 占位 token 换场景,anstok_v1 占位 token 全零场景,hardneg_v1 真场景,hardneg_v1 换场景,hardneg_v1 全零场景
相对距离,35.9,22.0,22.5,25.8,22.5,20.1
相对方向,34.6,37.2,34.0,40.4,41.2,41.2
路线规划,27.0,33.3,27.0,34.9,33.3,31.7
出现顺序,39.3,23.3,27.4,36.2,33.6,31.4
4 任务平均,34.2,28.9,27.7,34.3,32.7,31.1


相对方向的三个子类(test,占位 token):medium 在三种输入下相同。

In [5]:
pd.DataFrame({MODES[m]: {k: 100 * A[m]["vsibench_test_posed_f8"]["type/" + k] for k in MC["相对方向"]} for m in MODES}).round(1)

,真场景,换场景,全零场景
object_rel_direction_easy,49.2,50.0,50.0
object_rel_direction_medium,49.0,49.0,49.0
object_rel_direction_hard,17.7,16.0,25.5


## 2 留出集(6,786 题,训练没见过的场景)

长句答案按题目平均读,短答案按占位 token 读。hardneg_v1 没有占位 token,两种答案都按题目平均读。

In [6]:
def holdout(res, reading):
    return {MODES[m]: 100 * res[m]["holdout"][f"{reading}/all"]["acc"] for m in MODES}
rows = {"anstok_v1 长句(题目平均)": holdout(A, "long"), "anstok_v1 短答案(占位 token)": holdout(A, "short"),
        "anstok_v1 长句(占位 token)": holdout(A, "long_token"), "anstok_v1 短答案(题目平均)": holdout(A, "short_pooled"),
        "hardneg_v1 长句": holdout(H, "long"), "hardneg_v1 短答案": holdout(H, "short")}
df = pd.DataFrame(rows).T
df["真 − 换"] = df["真场景"] - df["换场景"]
df["随机"] = 100 * A["real"]["holdout"]["long/all"]["chance"]
df.round(1)

,真场景,换场景,全零场景,真 − 换,随机
anstok_v1 长句(题目平均),47.3,43.3,38.3,3.9,29.9
anstok_v1 短答案(占位 token),37.0,34.2,32.5,2.8,29.9
anstok_v1 长句(占位 token),42.0,39.3,36.5,2.6,29.9
anstok_v1 短答案(题目平均),34.7,33.2,30.0,1.6,29.9
hardneg_v1 长句,46.7,44.1,42.8,2.6,29.9
hardneg_v1 短答案,35.1,33.8,33.1,1.3,29.9


按题型,真场景 / 换场景:

In [7]:
keys = [k.split("/", 1)[1] for k in A["real"]["holdout"] if k.startswith("long/") and k.count("/") == 2]
cell = lambda r, k: f'{100 * r["real"]["holdout"][k]["acc"]:.1f} / {100 * r["shuffled"]["holdout"][k]["acc"]:.1f}' if k in r["real"]["holdout"] else ""
pd.DataFrame({"anstok_v1 长句": {k: cell(A, "long/" + k) for k in keys}, "hardneg_v1 长句": {k: cell(H, "long/" + k) for k in keys},
              "anstok_v1 短答案": {k: cell(A, "short/" + k) for k in keys}, "hardneg_v1 短答案": {k: cell(H, "short/" + k) for k in keys},
              "随机": {k: round(100 * A["real"]["holdout"]["long/" + k]["chance"], 1) for k in keys}})

,anstok_v1 长句,hardneg_v1 长句,anstok_v1 短答案,hardneg_v1 短答案,随机
spar/appearance_order,41.6 / 32.7,37.7 / 28.8,41.0 / 33.3,35.0 / 29.1,25.0
spar/spatial_imagination_oc_video_hard,44.7 / 43.3,44.8 / 43.7,33.7 / 31.3,32.3 / 33.6,31.2
spar/spatial_imagination_oo_video_hard,23.4 / 21.6,23.6 / 23.3,13.6 / 13.6,12.2 / 12.1,13.3
vsi590k/appearance_order,43.7 / 36.4,45.7 / 40.8,39.4 / 34.8,42.6 / 36.2,25.0
vsi590k/relative_count,55.8 / 38.3,55.2 / 45.5,44.2 / 37.0,21.4 / 21.4,33.3
vsi590k/relative_direction_object,45.8 / 44.0,46.1 / 45.8,46.5 / 46.1,46.5 / 45.6,37.3
vsi590k/relative_distance_object,44.4 / 39.7,42.5 / 41.4,34.4 / 31.2,30.9 / 31.0,25.0
vsi590k/relative_size_object,84.6 / 83.7,84.7 / 83.6,50.3 / 49.2,50.9 / 51.6,50.0


## 3 训练过程

验证集上同题错误答案的准确率(从 [正确答案, 它的错误答案] 中选对)。

In [8]:
log = [json.loads(l) for l in open("../results/hardneg_log.jsonl")]
rows = [r for r in log if "val_hard_acc" in r]
start = max(i for i, r in enumerate(rows) if r["step"] == rows[0]["step"])   # the file also holds a failed run; keep the last one
val = pd.DataFrame(rows[start:]).set_index("step")
(100 * val[["val_hard_acc", "val_short_hard_acc"]]).rename(columns={"val_hard_acc": "长句", "val_short_hard_acc": "短答案"}).round(1)

,长句,短答案
step,,
500,34.2,34.0
1000,36.0,36.0
1500,36.0,35.9
2000,37.1,39.0
2500,40.3,37.7
3000,43.4,38.3
3500,45.2,36.0
4000,49.5,38.6
4500,47.7,37.8


## 4 结论

1. 占位 token 加短答案目标在 VSI-Bench test 上高 1.9 个点,debiased 上没有差别。收益集中在相对距离(+10),它依赖场景。
2. 相对方向和空间想象没有变化:这两类在两种读法、两个数据集上都不随场景变化。把关系词单独作为目标,没有让模型去看场景里的关系。
3. 短答案的同题错误答案准确率低于长句(见第 3 节),相对大小的短答案在随机水平:短答案只有名字或数字,脱离题目后信息更少。

**运行记录**:正式训练提交了 4 次。两次在第 0 步通信超时(原因未查明,重提后正常);一次在第 2500 步后显存耗尽(80 GB),之后改为按长度分组编码答案文本。

**文件**:代码 `jepa3d/answer.py`、`jepa3d/model.py`(`answer_token`)、`jepa3d/trainer.py`;配置 `jepa3d/configs/anstok_v1.yaml`;结果 `experiments/anstok_v1/results/eval_final_otherscene.json`(修正的换场景)、`eval_final.json`(旧的换场景);checkpoint `experiments/anstok_v1/ckpt/final.pt`(11 GB)。